# 03 - Model Training

Stage 3 of 4. Reads the engineered matrices from `01_data_loading.ipynb`, builds the
train/test matrices, and fits every model in the project. No scoring happens here -
all metrics, tables and graphs live in `04_evaluation.ipynb`.

### Experiments trained here

| Group | Training data | Models |
|---|---|---|
| **A. Imbalanced** | Full `fraudTrain` (fraud ~0.58%) | Logistic Regression, Random Forest |
| **B. Balanced** | Random Under-Sampling to 1:1 | Logistic Regression, Random Forest |
| **C. Unsupervised** | Non-fraud rows only (no labels at fit) | Isolation Forest, One-Class SVM, Local Outlier Factor |

The supervised half of the supervised-vs-unsupervised comparison uses the **same two
models as group A** (identical parameters, identical training data), so they are fitted
once here and reused in notebook 04 under both names.

**Test set is never balanced** - all models are scored on the original imbalanced `fraudTest`.

## 1. Imports & setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.utils import resample

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42
NORMAL_SUBSAMPLE = 40_000  # ponytail: OCSVM/LOF too slow on full 1.29M normals
np.random.seed(RANDOM_STATE)

In [ ]:
from pathlib import Path

KAGGLE_IN = Path("/kaggle/input")


def find_data_dir():
    """Folder holding fraudTrain.csv / fraudTest.csv (Kaggle input mount or local archive/)."""
    for c in [KAGGLE_IN / "fraud-detection", Path("archive"), Path(".")]:
        if (c / "fraudTrain.csv").exists():
            return c
    for p in sorted(KAGGLE_IN.rglob("fraudTrain.csv")) if KAGGLE_IN.exists() else []:
        return p.parent
    return Path("archive")


# Raw CSVs: Kaggle dataset mount when on Kaggle, else the local archive/ folder.
DATA_DIR = find_data_dir()

# Everything this project generates goes into output/.
OUT = Path("/kaggle/working/output") if Path("/kaggle/working").exists() else Path("output")
for sub in ["data", "plots", "models", "preds", "results"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def upstream(rel):
    """Locate a file written by an earlier notebook (local run or Kaggle kernel input)."""
    local = OUT / rel
    if local.exists():
        return local
    if KAGGLE_IN.exists():
        for p in sorted(KAGGLE_IN.rglob(rel.split("/")[-1])):
            if p.as_posix().endswith("output/" + rel):
                return p
    raise FileNotFoundError("Run the earlier notebook first - missing: " + rel)


def save_fig(name):
    """Save the current matplotlib figure into output/plots/."""
    plt.savefig(OUT / "plots" / (name + ".png"), dpi=150, bbox_inches="tight")


print("DATA_DIR:", DATA_DIR)
print("OUT     :", OUT)

## 2. Load the engineered data from notebook 01

In [ ]:
train_fe = pd.read_parquet(upstream("data/train_fe.parquet"))
test_fe = pd.read_parquet(upstream("data/test_fe.parquet"))

print("Engineered train shape:", train_fe.shape)
print("Engineered test shape :", test_fe.shape)

## 3. Prepare X / y

Scale numeric features for Logistic Regression. Random Forest does not need scaling, but using the same matrix is fine.

In [ ]:
X_train_full = train_fe.drop(columns=["is_fraud"])
y_train_full = train_fe["is_fraud"].astype(int)

X_test = test_fe.drop(columns=["is_fraud"])
y_test = test_fe["is_fraud"].astype(int)

scaler = StandardScaler()
X_train_full_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_full),
    columns=X_train_full.columns,
    index=X_train_full.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

print("X_train_full:", X_train_full.shape, "| fraud in train:", int(y_train_full.sum()))
print("X_test      :", X_test.shape, "| fraud in test :", int(y_test.sum()))

## 4. Create balanced training set (Random Under-Sampling)

**Method:** keep **all fraud** rows; randomly sample the same number of non-fraud rows -> **1:1** balance.

**Important:** only the **training** set is balanced. Test set stays original/imbalanced.

In [ ]:
train_bal = train_fe.copy()
fraud = train_bal[train_bal["is_fraud"] == 1]
non_fraud = train_bal[train_bal["is_fraud"] == 0]

non_fraud_down = resample(
    non_fraud,
    replace=False,
    n_samples=len(fraud),
    random_state=RANDOM_STATE,
)

train_balanced = pd.concat([fraud, non_fraud_down]).sample(frac=1, random_state=RANDOM_STATE)

X_train_bal = train_balanced.drop(columns=["is_fraud"])
y_train_bal = train_balanced["is_fraud"].astype(int)

# Scale using the SAME scaler fitted on full train (no data leakage from test)
X_train_bal_scaled = pd.DataFrame(
    scaler.transform(X_train_bal),
    columns=X_train_bal.columns,
    index=X_train_bal.index,
)

print("Balanced train shape:", train_balanced.shape)
print(train_balanced["is_fraud"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["Not Fraud", "Fraud"], y_train_full.value_counts().sort_index().values, color=["#4C78A8", "#E45756"])
axes[0].set_title("Before: Imbalanced train")
axes[1].bar(["Not Fraud", "Fraud"], y_train_bal.value_counts().sort_index().values, color=["#4C78A8", "#E45756"])
axes[1].set_title("After: Balanced train (under-sample)")
plt.tight_layout()
save_fig("07_before_after_balancing")
plt.show()

## 5. Model factory & prediction helpers

In [ ]:
def make_models():
    """Same two models for both experiments."""
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=16,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return {"Logistic Regression": lr, "Random Forest": rf}


def anomaly_pred_score(model, X):
    y_pred = (model.predict(X) == -1).astype(int)
    y_score = -model.decision_function(X)  # higher = more anomalous
    return y_pred, y_score


test_pred = {"y_true": y_test.values}   # per-model predictions on the imbalanced test set
fitted = {}                             # name -> fitted estimator

## 6. Experiment A - train on **imbalanced** data

Models: **Logistic Regression** and **Random Forest**.

In [ ]:
models_imb = make_models()

print("Training Logistic Regression on FULL imbalanced train...")
models_imb["Logistic Regression"].fit(X_train_full_scaled, y_train_full)
name = "Imbalanced | Logistic Regression"
test_pred[name + "::pred"] = models_imb["Logistic Regression"].predict(X_test_scaled)
test_pred[name + "::score"] = models_imb["Logistic Regression"].predict_proba(X_test_scaled)[:, 1]
fitted[name] = models_imb["Logistic Regression"]
print("done.")

In [ ]:
print("Training Random Forest on FULL imbalanced train (may take a few minutes)...")
models_imb["Random Forest"].fit(X_train_full, y_train_full)
name = "Imbalanced | Random Forest"
test_pred[name + "::pred"] = models_imb["Random Forest"].predict(X_test)
test_pred[name + "::score"] = models_imb["Random Forest"].predict_proba(X_test)[:, 1]
fitted[name] = models_imb["Random Forest"]
print("done.")

## 7. Experiment B - train on **balanced** data

Same models, same test set. Only training distribution changes.

In [ ]:
models_bal = make_models()

print("Training Logistic Regression on BALANCED train...")
models_bal["Logistic Regression"].fit(X_train_bal_scaled, y_train_bal)
name = "Balanced | Logistic Regression"
test_pred[name + "::pred"] = models_bal["Logistic Regression"].predict(X_test_scaled)
test_pred[name + "::score"] = models_bal["Logistic Regression"].predict_proba(X_test_scaled)[:, 1]
fitted[name] = models_bal["Logistic Regression"]

print("Training Random Forest on BALANCED train...")
models_bal["Random Forest"].fit(X_train_bal, y_train_bal)
name = "Balanced | Random Forest"
test_pred[name + "::pred"] = models_bal["Random Forest"].predict(X_test)
test_pred[name + "::score"] = models_bal["Random Forest"].predict_proba(X_test)[:, 1]
fitted[name] = models_bal["Random Forest"]
print("done.")

## 8. Experiment C - unsupervised models

Fit on **non-fraud only** (labels are not used to learn the decision boundary).
Isolation Forest uses all normals; OCSVM / LOF use a 40k subsample.

In [ ]:
X_normal = X_train_full_scaled[y_train_full.values == 0]
rng = np.random.RandomState(RANDOM_STATE)
sub_idx = rng.choice(len(X_normal), size=min(NORMAL_SUBSAMPLE, len(X_normal)), replace=False)
X_normal_sub = X_normal.iloc[sub_idx]
print("Normals for IF:", len(X_normal), "| subsample for OCSVM/LOF:", len(X_normal_sub))

In [ ]:
print("Training Isolation Forest...")
iforest = IsolationForest(
    n_estimators=100, contamination=float(y_train_full.mean()), n_jobs=-1, random_state=RANDOM_STATE
)
iforest.fit(X_normal)
name = "Unsupervised | Isolation Forest"
test_pred[name + "::pred"], test_pred[name + "::score"] = anomaly_pred_score(iforest, X_test_scaled)
fitted[name] = iforest
print("done.")

In [ ]:
print("Training One-Class SVM...")
ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.01)
ocsvm.fit(X_normal_sub)
name = "Unsupervised | One-Class SVM"
test_pred[name + "::pred"], test_pred[name + "::score"] = anomaly_pred_score(ocsvm, X_test_scaled)
fitted[name] = ocsvm
print("done.")

In [ ]:
print("Training Local Outlier Factor...")
lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=float(y_train_full.mean()))
lof.fit(X_normal_sub)
name = "Unsupervised | Local Outlier Factor"
test_pred[name + "::pred"], test_pred[name + "::score"] = anomaly_pred_score(lof, X_test_scaled)
fitted[name] = lof
print("done.")

## 9. Training-set predictions for the overfitting check

Notebook 04 compares each supervised model on the data it was fitted on versus the test set.
Scoring the full 1.29M-row train matrix is unnecessary, so the imbalanced models are scored
on a 100k sample that keeps every fraud row.

In [ ]:
# ponytail: full 1.29M train predict can crash the kernel; 100k stratified sample is enough to see a train-test gap
rng = np.random.RandomState(RANDOM_STATE)
n_sample = min(100_000, len(y_train_full))
fraud_idx = np.flatnonzero(y_train_full.values == 1)
nonfraud_idx = np.flatnonzero(y_train_full.values == 0)
n_nf = n_sample - len(fraud_idx)
sample_idx = np.concatenate([fraud_idx, rng.choice(nonfraud_idx, size=n_nf, replace=False)])

X_tr_s = X_train_full_scaled.iloc[sample_idx]
X_tr = X_train_full.iloc[sample_idx]
y_tr = y_train_full.iloc[sample_idx]
print("Imbalanced TRAIN sample for scoring:", len(y_tr), "| fraud in sample:", int(y_tr.sum()))

train_sample_pred = {
    "y_true": y_tr.values,
    "Imbalanced | LR::pred": models_imb["Logistic Regression"].predict(X_tr_s),
    "Imbalanced | LR::score": models_imb["Logistic Regression"].predict_proba(X_tr_s)[:, 1],
    "Imbalanced | RF::pred": models_imb["Random Forest"].predict(X_tr),
    "Imbalanced | RF::score": models_imb["Random Forest"].predict_proba(X_tr)[:, 1],
}

train_bal_pred = {
    "y_true": y_train_bal.values,
    "Balanced | LR::pred": models_bal["Logistic Regression"].predict(X_train_bal_scaled),
    "Balanced | LR::score": models_bal["Logistic Regression"].predict_proba(X_train_bal_scaled)[:, 1],
    "Balanced | RF::pred": models_bal["Random Forest"].predict(X_train_bal),
    "Balanced | RF::score": models_bal["Random Forest"].predict_proba(X_train_bal)[:, 1],
}
print("Balanced TRAIN rows for scoring:", len(y_train_bal))

## 10. Save models and predictions

In [ ]:
pd.DataFrame(test_pred).to_parquet(OUT / "preds" / "test_preds.parquet", index=False)
pd.DataFrame(train_sample_pred).to_parquet(OUT / "preds" / "train_sample_preds.parquet", index=False)
pd.DataFrame(train_bal_pred).to_parquet(OUT / "preds" / "train_bal_preds.parquet", index=False)

# Feature importance from Random Forest (imbalanced train) - plotted in notebook 04
pd.Series(
    models_imb["Random Forest"].feature_importances_,
    index=X_train_full.columns,
    name="importance",
).to_csv(OUT / "models" / "rf_feature_importance.csv")

joblib.dump(scaler, OUT / "models" / "scaler.joblib", compress=3)
for mname, model in fitted.items():
    fname = mname.replace(" | ", "__").replace(" ", "_").replace("-", "") + ".joblib"
    joblib.dump(model, OUT / "models" / fname, compress=3)

print("Saved:")
for sub in ["preds", "models"]:
    for p in sorted((OUT / sub).iterdir()):
        print("  {}/{:<45} {:>8.1f} MB".format(sub, p.name, p.stat().st_size / 1e6))